In [0]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [0]:
import os
import sys

project_dir = "/Workspace/Users/ranayuvrajsingh1111@gmail.com/Olist ecomerce ETL"
if project_dir not in sys.path:
  sys.path.append(project_dir)

import utils
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# ==========================================
# 1. PARAMETERIZATION (Databricks Widget)
# ==========================================
dbutils.widgets.text("batch_name", "batch_1", "Target Batch Name")
batch_name = dbutils.widgets.get("batch_name")

print(f"Processing Silver Sellers for Batch: {batch_name}")

# ==========================================
# 2. EXTRACT INCOMING DATA
# ==========================================
df_sellers_raw = spark.read.table(
    "ecommerce_lakehouse.bronze.bronze_olist_sellers"
).filter(F.col("_batch_id") == batch_name)

# ==========================================
# 3. TRANSFORMATIONS & DATA HYGIENE
# ==========================================
df_cleaned = df_sellers_raw.select(
    F.trim(F.col("seller_id")).alias("seller_id"),
    F.coalesce(
        F.trim(F.col("seller_zip_code_prefix")), F.lit("00000")
    ).alias("seller_zip_code"),
    utils.clean_text_column("seller_city").alias(
        "seller_city"
    ),  # Using modular function
    F.upper(F.trim(F.col("seller_state"))).alias("seller_state"),
).dropDuplicates(["seller_id"])

# Compute Row Hash using SHA-256 for SCD Type 1 change detection
df_hashed = df_cleaned.withColumn(
    "row_hash",
    F.sha2(
        F.concat_ws(
            "||",
            F.coalesce(F.col("seller_zip_code"), F.lit("")),
            F.coalesce(F.col("seller_city"), F.lit("")),
            F.coalesce(F.col("seller_state"), F.lit("")),
        ),
        256,
    ),
)

# ==========================================
# 4. SURROGATE KEY GENERATION
# ==========================================
target_table_name = "ecommerce_lakehouse.silver.silver_sellers"

# Explicitly pass the value prefix ("S") and the column name ("seller_key")
df_incoming = utils.generate_surrogate_key(
    df=df_hashed,
    id_col="seller_id",
    prefix_val="S",
    key_col_name="seller_key",
    target_table_name=target_table_name,
    spark=spark,
)

# ==========================================
# 5. TARGET TABLE INITIALIZATION OR UPSERT (SCD TYPE 1)
# ==========================================
table_exists = spark.catalog.tableExists(target_table_name)
current_timestamp = F.current_timestamp()

if not table_exists:
  print("Target table does not exist. Initializing silver_sellers...")

  df_transformed = df_incoming.repartition(1).select(
      "seller_key",
      "seller_id",
      "seller_zip_code",
      "seller_city",
      "seller_state",
      "row_hash",
      current_timestamp.alias("_etl_inserted_at"),
      current_timestamp.alias("_etl_updated_at"),
      F.lit(False).alias("_delete_flag"),
  )

  df_transformed.write.format("delta").mode("overwrite").saveAsTable(
      target_table_name
  )
  print("Successfully initialized silver_sellers dimension!")

else:
  print("Table exists. Executing incremental upsert merge...")

  deltaTable = DeltaTable.forName(spark, target_table_name)

  deltaTable.alias("target").merge(
      source=df_incoming.alias("source"),
      condition="target.seller_id = source.seller_id",
  ).whenMatchedUpdate(
      condition="target.row_hash <> source.row_hash",
      set={
          "seller_zip_code": F.col("source.seller_zip_code"),
          "seller_city": F.col("source.seller_city"),
          "seller_state": F.col("source.seller_state"),
          "row_hash": F.col("source.row_hash"),
          "_etl_updated_at": current_timestamp,
      },
  ).whenNotMatchedInsert(
      values={
          "seller_key": F.col("source.seller_key"),
          "seller_id": F.col("source.seller_id"),
          "seller_zip_code": F.col("source.seller_zip_code"),
          "seller_city": F.col("source.seller_city"),
          "seller_state": F.col("source.seller_state"),
          "row_hash": F.col("source.row_hash"),
          "_etl_inserted_at": current_timestamp,
          "_etl_updated_at": current_timestamp,
          "_delete_flag": F.lit(False),
      },
  ).execute()

  print("Silver Sellers processing completed successfully!")

Processing Silver Sellers for Batch: batch_1
Target table does not exist. Initializing silver_sellers...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Successfully initialized silver_sellers dimension!
